# 2. AIND Ephys Preprocessing

Build an `AINDEPhysPreprocessingScanConfig`, expand it with `GridScanGenerationTask`, and run `AINDEPhysPreprocessingTask` for each coordinate. For every dispatched job the task applies the AIND preprocessing chain with spikeinterface (phase shift when the probe needs it, highpass filter, bad-channel detection and removal, common median reference), saves the result to `preprocessed_<name>/`, and estimates the probe's motion into `motion_<name>/`.

The toy recording is shorter than the pipeline's 120 s minimum, so `min_preprocessing_duration` is lowered, and it is clipped to 9 s.

## Imports

In [1]:
import json
from pathlib import Path

from obi_one.core.run_tasks import run_tasks_for_generated_scan
from obi_one.core.scan_generation import GridScanGenerationTask
from obi_one.scientific.tasks.aind_ephys._02_preprocessing.config import (
    AINDEPhysPreprocessingScanConfig,
)
from obi_one.scientific.tasks.aind_ephys.blocks import JobKwargs

## Build the scan config

`dispatch_output_path` is notebook 01's coordinate output folder.

In [2]:
obi_output = (Path.cwd() / "../../../../../../obi-output").resolve()
dispatch_output_path = obi_output / "01_aind_ephys_dispatch" / "grid_scan" / "0"
assert dispatch_output_path.exists(), f"{dispatch_output_path} not found. Run notebook 01 first."

scan_config = AINDEPhysPreprocessingScanConfig(
    initialize=AINDEPhysPreprocessingScanConfig.Initialize(
        dispatch_output_path=dispatch_output_path,
        min_preprocessing_duration=0.0,
        t_start=0.0,
        t_stop=9.0,
    ),
    job_kwargs=JobKwargs(n_jobs=1),
)

## Generate the grid scan and run the preprocessing task

In [3]:
grid_scan = GridScanGenerationTask(
    form=scan_config,
    output_root=obi_output / "02_aind_ephys_preprocessing" / "grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

run_tasks_for_generated_scan(grid_scan)

[PosixPath('/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-output/02_aind_ephys_preprocessing/grid_scan/0')]

## Load the preprocessed recording and the bad-channel labels

In [4]:
from collections import Counter

import spikeinterface as si

coord_dir = Path(grid_scan.single_configs[0].coordinate_output_root)
for binary_file in sorted(coord_dir.glob("binary_*.json")):
    name = binary_file.stem.removeprefix("binary_")
    recording = si.load(binary_file, base_folder=coord_dir)
    record = json.loads((coord_dir / f"data_process_preprocessing_{name}.json").read_text())
    print(name)
    print(" ", recording)
    print("  steps:", list(record["parameters"]["steps"]))
    print("  channel labels:", dict(Counter(record["outputs"]["channel_labels"])))
    print("  motion estimated:", (coord_dir / f"motion_{name}").is_dir())

block0_acquisition-ElectricalSeriesRaw_recording1
  BinaryFolderRecording: 31 channels - 25.0kHz - 1 segments - 225,001 samples - 9.00s 
                       float32 dtype - 26.61 MiB
  steps: ['highpass_filter', 'detect_and_remove_bad_channels', 'common_reference']
  channel labels: {'good': 31, 'dead': 1}
  motion estimated: True
